# Image Representation

A neural field represents an image as a function from pixel coordinates to
colour. This notebook fits one Kodak image, kodim17, under the image benchmark's
protocol: about 197k parameters each, and 1000 steps of Adam, each on the whole
image, with a cosine-annealed learning rate. It compares
FUTON, with a sinc and a cosine basis, against five representative baselines
of the benchmark: RFF (Fourier features), SIREN and FINER (periodic
activations), Instant-NGP (a hash grid) and TensoRF (a grid factorization).
Every setting is that of `configs/image.yaml`, which holds every model of the
benchmark.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.ticker import (
    FixedLocator,
    MaxNLocator,
    MultipleLocator,
    NullFormatter,
    StrMethodFormatter,
)
from matplotlib_inline.backend_inline import set_matplotlib_formats

import neurofield as nf

set_matplotlib_formats("retina")  # sharp figures on high-resolution screens
sns.set_theme(
    context="notebook", style="whitegrid", font_scale=0.85, rc={"grid.color": ".92"}
)

colorblind = sns.color_palette("colorblind")
# FUTON in black, its cosine basis dashed; the periodic activations in warm
# hues and the grids in cool ones.
PALETTE = {
    "FUTON (sinc)": "black",
    "FUTON (cosine)": "black",
    "RFF": colorblind[4],
    "SIREN": colorblind[1],
    "FINER": colorblind[3],
    "Instant-NGP": colorblind[2],
    "TensoRF": colorblind[0],
}
DASHES = {model: (4, 2) if model == "FUTON (cosine)" else "" for model in PALETTE}
# seaborn's own markers, each fixed to a model as its colour is
MARKERS = dict(zip(PALETTE, ["o", "X", (4, 0, 45), "P", (4, 0, 0), (4, 1, 0), "^"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

The plotting helpers: training time on a log axis, the metric on whole
decibels or half points, a summary table, images side by side, and
reconstructions above their errors.

In [ ]:
NICE = [1, 2, 5, 10]  # tick steps: 1, 2 or 5 times a power of ten


def time_axis(ax, times):
    """Training time on a log axis, ticked 1, 2, 5, 10, ... in plain numbers."""
    low, high = times.min() / 1.1, times.max() * 1.1
    ax.set(xscale="log", xlim=(low, high), xlabel="Training time (s)")
    for subs in ((1, 2, 5), (1,)):
        ticks = [m * 10.0**k for k in range(-3, 6) for m in subs]
        ticks = [tick for tick in ticks if low <= tick <= high]
        if len(ticks) <= 7:
            break
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())


def metric_axis(ax, metric, low, high):
    """The metric's axis over [low, high]: whole decibels for a PSNR, half
    points for an IoU, which cannot pass 100."""
    if metric == "IoU (%)":
        low = np.floor(2 * low) / 2
        ax.set_ylim(low, min(max(np.ceil(2 * high) / 2, low + 0.5), 100))
        ax.yaxis.set_major_locator(MultipleLocator(0.5))
    else:
        low = np.floor(low)
        ax.set_ylim(low, max(np.ceil(high), low + 1))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=NICE, integer=True))


def table(summary, formats, better):
    """The summary, each column in its format and, in each column of
    ``better`` ("higher" or "lower"), the best value bold and the second
    underlined."""

    def marks(column):
        rank = column.rank(method="min", ascending=better[column.name] == "lower")
        looks = {1: "font-weight: bold", 2: "text-decoration: underline"}
        return [looks.get(r, "") for r in rank]

    return (
        summary.style.format(formats, na_rep="")
        .apply(marks, subset=list(better))
        .hide(axis="index")
    )


def show(panels, ncols, width=11, **imshow):
    """Images side by side under their titles, in rows of ``ncols``, each
    drawn at the size of the first."""
    height, side = panels[0][1].shape[:2]
    nrows = -(-len(panels) // ncols)
    room = 0.45 if any("\n" in title for title, _ in panels) else 0.3
    fig, axes = plt.subplots(
        nrows,
        ncols,
        squeeze=False,
        layout="constrained",
        figsize=(width, nrows * (width / ncols * height / side + room)),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes.flat, panels):
        ax.imshow(image, interpolation="nearest", **imshow)
        ax.set_title(title, fontsize=9)
    return fig


def compare(panels, label, width=11):
    """The reference and the reconstructions in a row under their titles
    (``panels``, the reference first), and beneath each reconstruction its
    absolute error, averaged over the colour channels; the errors share one
    scale, which saturates at their 99th percentile, and its colour bar
    stands beneath the reference."""
    reference = panels[0][1].astype(float)
    errors = [np.abs(image - reference).mean(-1) for _, image in panels[1:]]
    height, side = reference.shape[:2]
    ncols = len(panels)
    fig, axes = plt.subplots(
        2,
        ncols,
        layout="constrained",
        figsize=(width, 2 * width / ncols * height / side + 0.45),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes[0], panels):
        ax.imshow(image, interpolation="nearest")
        ax.set_title(title, fontsize=9)
    vmax = np.percentile(np.concatenate([error.ravel() for error in errors]), 99)
    for ax, error in zip(axes[1, 1:], errors):
        maps = ax.imshow(
            error, cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
        )
    bar = fig.colorbar(maps, cax=axes[1, 0].inset_axes([0.8, 0.04, 0.07, 0.92]))
    bar.ax.yaxis.set_ticks_position("left")
    bar.ax.yaxis.set_label_position("left")
    bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=NICE))
    bar.ax.tick_params(labelsize=8)
    bar.set_label(label, fontsize=9)
    bar.outline.set_visible(False)
    return fig

## Data

The image, 768 by 512 pixels, and the region that the comparison at the end
magnifies.

In [ ]:
IMAGE_PATH = "../data/Kodak/kodim17.png"
CROP = (slice(104, 232), slice(264, 392))  # rows and columns: the laurel wreath

image = nf.ImageCoordinateDataset(IMAGE_PATH)  # every pixel, for evaluation
H, W = image.grid_size
original = image.original.permute(1, 2, 0).numpy()  # (H, W, 3), uint8

fig = show(
    [("kodim17", original), ("Magnified region", original[CROP])], ncols=2, width=6
)
rows, columns = CROP
box = plt.Rectangle(
    (columns.start, rows.start), columns.stop - columns.start, rows.stop - rows.start
)
fig.axes[0].add_patch(box).set(fill=False, edgecolor="red", linewidth=1.5)
plt.show()

## Models

Each entry gives a model's class, its arguments and its learning rate.
`# deviation` marks a setting that departs from the authors' code, with the
reason.

In [ ]:
CONFIGS = [
    {
        "name": "RFF",  # Tancik et al. (NeurIPS 2020), 2D image regression
        "class": nf.RFF,
        "kwargs": {
            "hidden_features": 212,
            "hidden_layers": 3,
            "num_frequencies": 256,
            # deviation: tuned on four images at the rate below, where Tancik's 10
            # on [0, 1] coordinates, 5 here, costs 0.9 dB and 20 costs 0.7
            "sigma": 10.0,
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,  # 1e-3 costs 0.9 dB and 1e-2 6.6 on two images
    },
    {
        "name": "SIREN",  # FINER image benchmark
        "class": nf.SIREN,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,
    },
    {
        "name": "FINER",  # FINER image benchmark
        "class": nf.FINER,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 4,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,
    },
    {
        "name": "Instant-NGP",  # instant-ngp configs/image/base.json
        "class": nf.InstantNGP,
        "kwargs": {
            "num_levels": 20,  # deviation: 16 in the reference; 20 meets the budget
            "features_per_level": 2,
            "log2_hashmap_size": 13,
            "base_resolution": 16,
            "max_resolution": max(H, W) // 2,
            "hidden_features": 64,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        # deviation: the reference 1e-2 costs 0.9 dB on two images, 1e-1 8.1
        "lr": 3e-2,
    },
    {
        "name": "TensoRF",
        "class": nf.TensoRF,
        "kwargs": {
            "rank": 176,
            "resolution": 256,
            "mode": "cp",
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON (sinc)",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (  # on the pixels
                "sinc",
                {"num_components": (H // 2, W // 2), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": 280}),
            "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON (cosine)",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (  # on the pixels
                "cosine",
                {"num_components": (H // 2, W // 2), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": 280}),
            "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-1,
    },
]

## Training

Every model is compiled with `torch.compile`, and an untimed forward and
backward pass compiles it before training starts. The training times exclude
that compilation and the evaluations.

In [ ]:
METRICS = {"psnr": nf.psnr, "ssim": nf.ssim, "lpips": nf.lpips}
train_dataset = nf.ImageCoordinateDataset(IMAGE_PATH)  # the whole image at every step
results = {}

for config in CONFIGS:
    torch.manual_seed(0)
    model = config["class"](in_features=2, out_features=3, **config["kwargs"])
    print(f"{config['name']}: {nf.count_parameters(model):,} parameters")
    model.to(device).compile()
    model(train_dataset[0]["input"][None].to(device)).sum().backward()  # compiles it
    model.zero_grad(set_to_none=True)
    result = nf.train(
        model,
        train_dataset,
        image,
        lr=config["lr"],
        num_steps=1000,
        metrics=METRICS,
        log_interval=50,
        eval_interval=50,
        device=device,
        **config.get("train_kwargs", {}),
    )
    output = nf.chunked_inference(model, image.input, device=device)
    result["reconstruction"] = image.postprocess(output.cpu()).permute(1, 2, 0).numpy()
    results[config["name"]] = result

    del model, result["model_dict"]
    torch.cuda.empty_cache()
    torch.compiler.reset()  # the next model compiles anew

## Convergence

The PSNR of the whole image, every 50 steps.

FUTON is black, its cosine basis dashed; the periodic activations have warm
hues and the grids cool ones.

In [ ]:
records = pd.DataFrame(
    {
        "Model": name,
        "Step": entry["step"],
        "Time (s)": entry["elapsed"],
        "PSNR (dB)": entry["eval"]["psnr"],
    }
    for name, result in results.items()
    for entry in result["history"]
    if "eval" in entry
)

order = [model for model in PALETTE if model in set(records["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, x in zip(axes, ("Step", "Time (s)")):
    sns.lineplot(
        records,
        x=x,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        dashes=DASHES,
        legend=ax is axes[1],
        ax=ax,
    )
axes[0].xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
final = records.groupby("Model")["PSNR (dB)"].last()
low, high = final.min(), records["PSNR (dB)"].max()
margin = 0.05 * (high - low)  # so that no curve ends on the frame
metric_axis(axes[0], "PSNR (dB)", low - margin, high + margin)
shown = records["PSNR (dB)"] >= axes[0].get_ylim()[0]  # the evaluations in view
time_axis(axes[1], records.loc[shown, "Time (s)"])

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Summary

Throughput is the median rate of the evaluations, which reconstruct the whole
image, and inference memory their median peak; training memory is the peak
of training. In each column the best value is bold and the second
underlined.

In [ ]:
rows = []
for name, result in results.items():
    evaluations = [entry["eval"] for entry in result["history"] if "eval" in entry]
    duration = np.median([e["duration"] for e in evaluations])
    memory = np.median([e["peak_memory"] for e in evaluations])
    rows.append(
        {
            "Model": name,
            "Params (k)": result["config"]["num_params"] / 1e3,
            "Training time (s)": result["history"][-1]["elapsed"],
            "Training memory (MB)": result["config"]["peak_memory"] / 1e6,
            "Throughput (MPix/s)": H * W / duration / 1e6,
            "Inference memory (MB)": memory / 1e6,
            "PSNR (dB)": evaluations[-1]["psnr"],
            "SSIM": evaluations[-1]["ssim"],
            "LPIPS": evaluations[-1]["lpips"],
        }
    )
summary = pd.DataFrame(rows).sort_values("PSNR (dB)", ascending=False)

formats = {
    "Params (k)": "{:.1f}",
    "Training time (s)": "{:.1f}",
    "Training memory (MB)": "{:.0f}",
    "Throughput (MPix/s)": "{:.1f}",
    "Inference memory (MB)": "{:.0f}",
    "PSNR (dB)": "{:.2f}",
    "SSIM": "{:.3f}",
    "LPIPS": "{:.3f}",
}
better = {
    "Training time (s)": "lower",
    "Training memory (MB)": "lower",
    "Throughput (MPix/s)": "higher",
    "Inference memory (MB)": "lower",
    "PSNR (dB)": "higher",
    "SSIM": "higher",
    "LPIPS": "lower",
}
table(summary, formats, better)

In [ ]:
order = [model for model in PALETTE if model in set(summary["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, cost in zip(axes, ("Training time (s)", "Throughput (MPix/s)")):
    sns.scatterplot(
        summary,
        x=cost,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        markers=MARKERS,
        s=70,
        legend=ax is axes[1],
        ax=ax,
    )
    ax.margins(x=0.08)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].set_xlabel("Training time (s) ↓")
axes[1].set_xlabel("Throughput (MPix/s) ↑")
low, high = summary["PSNR (dB)"].min(), summary["PSNR (dB)"].max()
metric_axis(axes[0], "PSNR (dB)", low - 0.08 * (high - low), high + 0.08 * (high - low))

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Qualitative Comparison

The magnified region of every reconstruction, titled with its PSNR on the whole
image, and beneath, the error of each, averaged over the colour channels, on
one scale.

In [ ]:
crops = [("Original", original[CROP])] + [
    (f"{name}\n{psnr:.2f} dB", results[name]["reconstruction"][CROP])
    for name, psnr in zip(summary["Model"], summary["PSNR (dB)"])
]
compare(crops, "Error (8-bit levels)")
plt.show()